In [1]:
import os
import sys
import pandas as pd
import time

module_path = os.path.abspath(os.path.join('../../music-sources-unified'))
if module_path not in sys.path: sys.path.append(module_path)
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path: sys.path.append(module_path)
    
import unify_lib as uni
from ytmusic_library import YTMusicPlaylists

HEADER_FILE = '../oauth.json'
BACKUP_DIR = '../playlists/'
Y = YTMusicPlaylists(header=HEADER_FILE, playlist_tsv_dir=BACKUP_DIR)
print(f"Loaded {len(Y.playlists['title'].unique())} playlists")


Using header file: ../oauth.json
Loaded 462 playlists


## Like not liker miner

Get all ytmusic entries, get like entries and aget not like entries, create artist - track key and move all matching likes and not like to respective playlist

In [2]:
need_review = Y.get_like_not_like_tracks_to_review()
# need_review.to_csv(Y.need_rate_tsv, sep='\t', index=True)
# Note: now part of ytmusic backup

# v2.b
# Loaded 38329 like, 6565 not like entries, and 151048 total tracks
# Loaded 35452 like, 6565 not like entries, that have an entry in ALL_TRACKS
# Processing LIKE tracks...
#  Found 742 new tracks to LIKE
#  Found 165 tracks to LIKE but already in NOT LIKE
# Processing NOT LIKE tracks...
#  Found 445 new tracks to NOT LIKE
#  Found 202 tracks to NOT LIKE but they are already in LIKE
# Loaded 7734 manually labeled entries, 2839 are LIKE, 4850 NOT_LIKE
# Reduced new LIKE from 742 to 480 entries after removing reviewed matches
# Reduced new NOT_LIKE from 445 to 254 entries after removing reviewed matches
# Reduced skip_not_like from 165 to 157 entries after removing reviewed matches
# Reduced skip_is_like from 202 to 201 entries after removing reviewed matches
# Keeping 6331 not like after remove LIKE

# v2.a (olderversions had dupes in Like)
# Loaded 38329 like, 4996 not like entries, and 163918 total tracks
# Keeping 4996 not like after remove LIKE
# Saving _need_like tsv with 2868 entries
# Saving _need_like_but_is_not_like tsv with 187 entries
# Saving _need_not_like tsv with 4749 entries
# Saving _need_not_like_but_is_like tsv with 259 entries

# v1
# Loaded 41461 like, 4806 not like entries, and 149082 total tracks
# Keeping 4613 not like after remove LIKE
# v0
# Loaded 39646 like, 4531 not like entries, and 149039 total tracks
# Keeping 4383 not like after remove LIKE


Loaded 38329 like, 6565 not like entries, and 151048 total tracks
Loaded 6565 not like entries, that have an entry in ALL_TRACKS
Keeping 6331 not like after remove LIKE
Processing LIKE tracks...
 Found 742 new tracks to LIKE
 Found 165 tracks to LIKE but already in NOT LIKE

Processing NOT LIKE tracks...
 Found 445 new tracks to NOT LIKE
 Found 202 tracks to NOT LIKE but they are already in LIKE

Top 10 playlists impacted:
decoded_list
z__thumbs_up_like    48
nan                  46
y_2005s_thumbs_up    45
Liked Music          33
indie                32
psych rock modern    30
y_2013_thumbs_up     28
y_2012_thumbs_up     27
y_2014_thumbs_up     24
electronic           21
Name: count, dtype: int64
Loaded 7734 manually labeled entries, 2839 are LIKE, 4850 NOT_LIKE
Reduced new LIKE from 742 to 480 entries after removing reviewed matches
Reduced new NOT_LIKE from 445 to 254 entries after removing reviewed matches
Reduced skip_not_like from 165 to 157 entries after removing reviewed matches

## Do this after manually reviewing above

In [ ]:
# use this to remove tracks already rated, then add ones that need rating to end or something
manual_picks = pd.read_csv(Y.manual_rate_tsv, sep='\t', index_col=0).drop_duplicates(keep='first') # remove duplicate index
to_like = manual_picks.loc[manual_picks['manual_rating'] == 'LIKE']
to_not_like = manual_picks.loc[manual_picks['manual_rating'] == 'NOT_LIKE']
# Loaded 7734 manually labeled entries, 2839 are LIKE, 4850 NOT_LIKE
print(f'Loaded {len(manual_picks)} manually labeled entries, {len(to_like)} are LIKE, {len(to_not_like)} NOT_LIKE')


to_like_pl_id = Y.yt.create_playlist(
  title='_likes_new', video_ids=list(to_like.index), privacy_status='PRIVATE',
  description=f'{len(to_like)} tracks that should be like')
to_not_like_pl_id = Y.yt.create_playlist(
  title='_not_likes_new', video_ids=list(to_not_like.index), privacy_status='PRIVATE',
  description=f'{len(to_not_like)} tracks that should be not like')


# Like the to_like playlist
# Playlist _likes_new: Rated 2708 of 2839 tracks as LIKE
Y.playlist_rate_all_songs(Y.playlist_get_info(to_like_pl_id), 'LIKE', sleep_time=0.5,  verbose=False, skip_if_dislike=False)

# delete to like
# merge not like with xx? or split into 2 art?


## Fix existing playlists

In [170]:
not_like_df = pd.read_csv(Y.not_like_tsv, sep='\t', index_col=0)
not_like_vids = frozenset(not_like_df.videoId)
print(f'Loaded {len(not_like_vids)} not like entries')
radio_to_like_df = Y._radio_to_like_map.copy()
no_matchlike_pl = radio_to_like_df.loc[radio_to_like_df['like_playlist'].isna()]
print(f'Loaded {len(radio_to_like_df)} radio to like playlist matches, {len(no_matchlike_pl)} do not have a match')
completed = {}
# Loaded 4806 not like entries
# Loaded 181 radio to like playlist matches, 28 do not have a match

Loaded 4806 not like entries
Loaded 181 radio to like playlist matches, 28 do not have a match


### TODO integrate new functions in Y which were copied from here, also move this notbook back to ytmusisc (and maybe some assets)

In [ ]:
VERBOSE = True
REMOVE_NOT_LIKE_AND_DISLIKE = True
MOVE_LIKE = True
MIN_NUM_LIKE = 10
CREATE_LIKE_PLAYLIST = True

counter_df = []
like_not_like_vids = []
for pl in Y.playlists.itertuples():
  if 'radio' not in pl.title: continue

  if pl.title in completed: 
    if VERBOSE: print(f'Already completed {pl.title}')
    continue

  like_pl = radio_to_like_df.loc[radio_to_like_df['radio_playlist'] == pl.title].iloc[0]['like_playlist']

  pl_info = Y.playlist_get_info(pl.playlistId, use_cache=True)
  remove_tracks = []
  move_like_tracks = []
  pl_counters = {'name': pl.title, 'removed_dislike': 0, 'moved_like': 0, 'removed_not_like': 0, 'like_and_not_like': 0}
  for track in pl_info.get('tracks', []):
    if track['likeStatus'] == 'DISLIKE':
      pl_counters['removed_dislike'] += 1
      remove_tracks.append(track)
    elif track['likeStatus'] == 'LIKE':
      pl_counters['moved_like'] += 1
      move_like_tracks.append(track)
      if track['videoId'] in not_like_vids:
        pl_counters['like_and_not_like'] += 1
        like_not_like_vids.append(track['videoId'])
    elif track['videoId'] in not_like_vids:
      pl_counters['removed_not_like'] += 1
      remove_tracks.append(track)     
  counter_df.append(pl_counters)
  if VERBOSE: print(100*'*' + f'\n{pl_counters}')
  
  # Handle flagged tracks
  if MOVE_LIKE and len(move_like_tracks) > MIN_NUM_LIKE:
    # Create like playlist and add from 'move_like_tracks'
    like_pl_id = None
    like_vids = [t['videoId'] for t in move_like_tracks]
    if pd.isna(like_pl):
      if CREATE_LIKE_PLAYLIST:
        like_pl = pl.title.replace('radio', 'like')
        like_pl_id = Y.yt.create_playlist(
            title=like_pl,  description=f'Created for dumping likes from {pl.title}',
            privacy_status='PRIVATE', video_ids=like_vids)
        if VERBOSE: print(f'Created LIKE playlist for {pl.title}: {like_pl}')
        time.sleep(3)

    else:
      like_pl_id = Y.query_by_title(like_pl).playlistId
      like_orig_vids = frozenset([t['videoId'] for t in Y.playlist_get_info(like_pl_id, use_cache=False).get('tracks', [])])
      like_new_vids = frozenset(like_vids) - like_orig_vids
      like_dedupe_num = len(like_vids) - len(like_new_vids)
      if like_dedupe_num > 0:
        like_vids = list(like_new_vids)
        
      if len(like_vids):
        status = Y.yt.add_playlist_items(playlistId=like_pl_id, videoIds=like_vids, duplicates=False)
        # Somtimes this still fails, fallback is to reemove like pl mapping so it generates a fresh pl
        assert status['status'] == 'STATUS_SUCCEEDED', f'Bad Status for {pl.title} add {len(move_like_tracks)} LIKE tracks: {status}'
        time.sleep(1)
      elif VERBOSE: 
        print(f'No new LIKE tracks to add to playlist {like_pl_id}')
        
    if like_pl_id == None:
      if VERBOSE: print(f'No LIKE playlist for {pl.title}, so not moving {len(move_like_tracks)} LIKE tracks')
      continue
    if VERBOSE: print(f'Added {len(move_like_tracks)} LIKE entries from {pl.title} to {like_pl_id}')
    
    status = Y.yt.remove_playlist_items(pl.playlistId, move_like_tracks)
    assert str(status) == 'STATUS_SUCCEEDED', f'Bad Status for {pl.playlistId} remove {len(move_like_tracks)} LIKE tracks: {status}'
    time.sleep(1)
    if VERBOSE: print(f'Moved {len(move_like_tracks)} LIKE entries from {pl.title}')
    
  if REMOVE_NOT_LIKE_AND_DISLIKE and len(remove_tracks):
    status = Y.yt.remove_playlist_items(pl.playlistId, remove_tracks)
    assert str(status) == 'STATUS_SUCCEEDED', f'Bad Status for {pl.playlistId} remove {len(remove_tracks)} NOT LIKE tracks: {status}'
    if VERBOSE: print(f'Removed {len(remove_tracks)} NOT_LIKE entries from {pl.title}')    
    time.sleep(1)
  completed[pl.title] = pl_counters
  

complete_df = pd.DataFrame(completed).T[['removed_dislike', 'moved_like', 'removed_not_like', 'like_and_not_like', 'status']].sort_values('moved_like')
complete_df['total_changes'] = complete_df[['removed_dislike', 'moved_like', 'removed_not_like', 'like_and_not_like']].sum(axis=1)
complete_df = complete_df.sort_values('total_changes', ascending = False)
complete_df.to_csv('../logs/ytmusic_match_like_not_like_result.tsv', sep='\t')


### Count number of tracks in radio playlists
(takes 5m)

In [174]:
playlists = Y.get_playlist_counts(verbose=False, filter_title='radio')
radio_counts_df = playlists.loc[playlists.title.str.contains('radio')].sort_values('track_count')
radio_counts_df = radio_counts_df[['title', 'track_count', 'duration_hours', 'privacy', 'playlist_id']]
radio_counts_df.to_csv(Y.radio_count_file, sep='\t', index=False)
radio_counts_df
# last run 7-2023

,title,playlist_id,track_count,privacy,duration_hours
67,Reggae 1970 roots radio,PLWptjpDqazOxNFmvqxnS9RYTsm51yErv6,0,PRIVATE,0
0,ambient Indie Synths radio,PLWptjpDqazOw7al5TJVXbaiQr1ILDBk_y,27,PRIVATE,2
44,Jazz Feels the Blues radio,PLWptjpDqazOwjAXRmi4n6wLMiV9hsm1pT,27,PRIVATE,3
82,Soul Food Kitchen radio,PLWptjpDqazOxDwaPfQXeeho7O2tLmgjP_,28,PRIVATE,2
71,rock 1967 Monterey Pop Festival radio,PLWptjpDqazOyoXjkoYrtOq-HxIFikPMb0,33,PRIVATE,2
...,...,...,...,...,...
87,x_r.2010smusic_tracks_radio,PLWptjpDqazOyU9d_yupbcHOq-9MYYOonW,1106,PRIVATE,72
122,x_r.futuregarage_tracks_radio,PLWptjpDqazOxj9ct9vG8tMeOMdj2rH_PQ,1142,PRIVATE,97
150,x_r.reggae_tracks_radio,PLWptjpDqazOxFpTJUsBAOp163DArs5Wyf,1198,PRIVATE,83
69,Reggae radio,PLWptjpDqazOwE761BnO1IfHJxwd9W8waZ,1351,PRIVATE,93
